
# 08 — Winger Player Profiles

## DECODING WINGER PLAYSTYLES

### Research Objective

Develop a reproducible, interpretable player-profiling
framework that combines behavioural statistics,
continuous playstyle dimensions, historical player
similarity and descriptive archetype membership.

## Research Question

How can a winger's attacking behaviour be represented
in a way that allows meaningful comparison with the
historical winger population and other footballers?

## Data

The analysis covers:

- 1,396 player-club-season observations.
- 665 distinct footballers.
- Europe's Big Five leagues.
- Seasons 2017–18 through 2023–24.
- Minimum 1,125 minutes per qualifying observation.
- 20 attacking-behaviour features across eight families.

The observational unit is a player-club-season spell.

Different seasons of the same footballer are treated
as potentially different behavioural observations.

## Existing Analytical Foundations

Notebook 04:
20-feature attacking-behaviour representation,
raw measurements, global standardization and
behavioural-family definitions.

Notebook 05:
Principal Component Analysis identifying major
continuous dimensions of winger attacking behaviour.

Notebook 06:
Euclidean nearest-neighbour similarity in the
full standardized 20-dimensional feature space,
with distinct-footballer candidate selection.

Notebook 07:
Five exploratory K-Means groups representing
interpretable attacking-behaviour profiles,
with documented assignment ambiguity and
robustness limitations.

## Profiling Principles

1. Describe playing style, not player quality.

2. Do not interpret higher feature percentiles
   as inherently better.

3. Preserve the full 20-dimensional representation
   for primary similarity calculations.

4. Use PCA for continuous style interpretation.

5. Treat K=5 archetype labels as descriptive
   rather than natural, exclusive player categories.

6. Report assignment ambiguity and prototype distance.

7. Distinguish observation-level results from
   career-wide conclusions.

8. Reuse existing validated models rather than
   silently refitting them.

## Expected Deliverables

- Player-level behavioural fingerprint.
- PCA style-position summary.
- Archetype comparison and explanation.
- Historical stylistic-neighbour report.
- Reusable player-profiling functions.
- Reproducible exported profiling artifacts.

In [ ]:
# ============================================================
# NOTEBOOK 08 — SETUP
# Files uploaded manually to Google Colab /content/
# No upload widget, Drive mount or folder creation
# ============================================================

from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import pairwise_distances

DATA_DIR = Path("/content")

required_files = {
    "raw": "winger_style_features_raw.parquet",
    "scaled": "winger_style_features_scaled.parquet",
    "specification": "winger_style_feature_specification.csv",
    "neighbours": "top10_primary_euclidean.parquet",
    "assignments": "winger_k5_profile_assignments.parquet",
    "centroids": "winger_k5_centroids_standardized.csv",
    "model": "winger_kmeans_primary_k5.joblib",
    "manifest": "winger_clustering_manifest.json"
}

missing = []

print("=" * 65)
print("NOTEBOOK 08 — INPUT VALIDATION")
print("=" * 65)

for key, filename in required_files.items():

    path = DATA_DIR / filename

    if path.is_file():
        print(f"[FOUND]   {filename}")
    else:
        print(f"[MISSING] {filename}")
        missing.append(filename)

if missing:
    raise FileNotFoundError(
        "Upload the missing files manually to /content/:\n"
        + "\n".join(missing)
    )

print("\nAll required inputs found.")

In [ ]:
# ============================================================
# LOAD NOTEBOOK 04 + NOTEBOOK 07 DATA
# ============================================================

raw_df = pd.read_parquet(
    DATA_DIR / required_files["raw"]
).reset_index(drop=True)

scaled_df = pd.read_parquet(
    DATA_DIR / required_files["scaled"]
).reset_index(drop=True)

feature_spec = pd.read_csv(
    DATA_DIR / required_files["specification"]
)

assignments = pd.read_parquet(
    DATA_DIR / required_files["assignments"]
).reset_index(drop=True)

centroids = pd.read_csv(
    DATA_DIR / required_files["centroids"]
)

with open(
    DATA_DIR / required_files["manifest"],
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

# ------------------------------------------------------------
# Validate feature specification
# ------------------------------------------------------------

features = feature_spec["feature"].tolist()

assert len(features) == 20
assert len(set(features)) == 20

assert "progressive_distance_per_carry" not in features

assert all(f in raw_df.columns for f in features)
assert all(f in scaled_df.columns for f in features)

assert manifest["feature_order"] == features
assert manifest["feature_count"] == 20

# ------------------------------------------------------------
# Validate observation counts
# ------------------------------------------------------------

n = len(raw_df)

assert n == 1396
assert len(scaled_df) == n
assert len(assignments) == n

assert (
    assignments["observation_row_idx"].to_numpy()
    == np.arange(n)
).all()

assert scaled_df["player_id"].nunique() == 665

# ------------------------------------------------------------
# Validate identity and row alignment
# ------------------------------------------------------------

identity_cols = [
    "player",
    "player_id",
    "squad",
    "season"
]

for col in identity_cols:

    assert col in raw_df.columns
    assert col in scaled_df.columns
    assert col in assignments.columns

    raw_values = raw_df[col].astype(str).to_numpy()
    scaled_values = scaled_df[col].astype(str).to_numpy()
    assignment_values = assignments[col].astype(str).to_numpy()

    assert np.array_equal(
        raw_values, scaled_values
    ), f"Raw/scaled identity mismatch: {col}"

    assert np.array_equal(
        scaled_values, assignment_values
    ), f"Assignment identity mismatch: {col}"

# ------------------------------------------------------------
# Validate feature matrices
# ------------------------------------------------------------

X_raw = raw_df[features].to_numpy(dtype=float)
X_scaled = scaled_df[features].to_numpy(dtype=float)

assert X_raw.shape == (1396, 20)
assert X_scaled.shape == (1396, 20)

assert np.isfinite(X_raw).all()
assert np.isfinite(X_scaled).all()

# ------------------------------------------------------------
# Family specification
# ------------------------------------------------------------

family_map = feature_spec.set_index(
    "feature"
)["macro_family"]

print("=" * 65)
print("DATASET VALIDATION PASSED")
print("=" * 65)

print("Player spells:", n)
print("Distinct footballers:", scaled_df["player_id"].nunique())
print("Behavioural features:", len(features))
print("Behavioural families:", family_map.nunique())

print("\nFeature-family distribution:")

display(
    feature_spec.groupby("macro_family")
    .size()
    .rename("n_features")
    .reset_index()
)

In [ ]:
# ============================================================
# VERIFY CLUSTERING AND SIMILARITY ARTIFACTS
# ============================================================

k5_model = joblib.load(
    DATA_DIR / required_files["model"]
)

neighbours = pd.read_parquet(
    DATA_DIR / required_files["neighbours"]
)

# ------------------------------------------------------------
# 1. Reproduce original K=5 assignments
# ------------------------------------------------------------

predicted_clusters = k5_model.predict(X_scaled)

saved_clusters = assignments[
    "k5_cluster"
].to_numpy()

np.testing.assert_array_equal(
    predicted_clusters,
    saved_clusters
)

assert k5_model.cluster_centers_.shape == (5, 20)

# Compare saved centroids to fitted model
centroids_sorted = (
    centroids.sort_values("cluster")
    .reset_index(drop=True)
)

assert centroids_sorted["cluster"].tolist() == list(range(5))

np.testing.assert_allclose(
    centroids_sorted[features].to_numpy(dtype=float),
    k5_model.cluster_centers_,
    rtol=1e-6,
    atol=1e-6
)

print("[PASS] K=5 assignments and centroids reproduced.")

# ------------------------------------------------------------
# 2. Validate PCA scores
# ------------------------------------------------------------

pc_cols = ["PC1", "PC2", "PC3", "PC4", "PC5"]

assert all(c in assignments.columns for c in pc_cols)

assert np.isfinite(
    assignments[pc_cols].to_numpy(dtype=float)
).all()

print("[PASS] Five PCA coordinates available.")

# ------------------------------------------------------------
# 3. Validate Notebook 06 neighbour structure
# ------------------------------------------------------------

needed_neighbour_cols = [
    "target_row_idx",
    "candidate_row_idx",
    "rank",
    "distance",
    "candidate_player",
    "candidate_player_id",
    "candidate_squad",
    "candidate_season"
]

assert all(
    c in neighbours.columns
    for c in needed_neighbour_cols
)

assert len(neighbours) == n * 10

assert (
    neighbours.groupby("target_row_idx").size() == 10
).all()

assert (
    neighbours["target_row_idx"].nunique() == n
)

assert set(neighbours["target_row_idx"]) == set(range(n))

assert (
    neighbours.groupby("target_row_idx")["rank"]
    .apply(lambda s: set(s) == set(range(1, 11)))
).all()

assert neighbours["candidate_row_idx"].between(
    0, n - 1
).all()

assert (
    neighbours.groupby("target_row_idx")[
        "candidate_player_id"
    ].nunique() == 10
).all()

# ------------------------------------------------------------
# 4. Recompute stored Euclidean distances
# ------------------------------------------------------------

target_idx = neighbours[
    "target_row_idx"
].to_numpy(dtype=int)

candidate_idx = neighbours[
    "candidate_row_idx"
].to_numpy(dtype=int)

recomputed_distances = np.linalg.norm(
    X_scaled[target_idx] - X_scaled[candidate_idx],
    axis=1
)

np.testing.assert_allclose(
    recomputed_distances,
    neighbours["distance"].to_numpy(dtype=float),
    rtol=1e-6,
    atol=1e-6
)

# No footballer should be their own similarity candidate
target_player_ids = (
    scaled_df.iloc[target_idx]["player_id"]
    .astype(str)
    .to_numpy()
)

candidate_player_ids = (
    scaled_df.iloc[candidate_idx]["player_id"]
    .astype(str)
    .to_numpy()
)

assert np.all(
    target_player_ids != candidate_player_ids
)

print("[PASS] All 13,960 neighbour distances reproduced.")
print("[PASS] No same-footballer neighbours.")
print("[PASS] Unique candidate footballers per target.")

print("\nAll Notebook 08 inputs are consistent.")

## 2. Behavioural Fingerprint

A winger's behavioural fingerprint summarizes their
attacking actions relative to the historical winger
population.

Three complementary measurements are retained:

### Raw Feature Value

The original measurement in its defined unit.

Examples include progressive carries per 90,
take-ons per 100 live touches and passing shares.

Raw values preserve football meaning but cannot
be compared directly across different statistical
units.

### Standardized Feature Value

The deviation from the population mean, measured
in standard deviations.

Positive values indicate above-population-average
measurements, while negative values indicate
below-population-average measurements.

This describes relative behaviour, not player quality.

### Population Percentile

The player's relative position within the historical
distribution of the same feature.

A 90th-percentile value indicates that the observation
ranks relatively high on that particular statistic.

It does not imply the player is better than 90% of
other footballers.

Percentiles are calculated across the 1,396
player-club-season observations.

Because some footballers contribute multiple seasons,
these are observation-weighted population percentiles,
not equally weighted footballer percentiles.

### Interpretation

No single feature adequately describes a winger's style.

Features must be interpreted together, with attention
to their attacking-behaviour families, units and
potential overlap.

Higher values are not automatically desirable.

For example, higher average shot distance means a
player typically shoots from farther away, while
higher crossing share indicates greater reliance on
crossing as an attacking action.

The initial fingerprint will be examined as a
20-feature statistical description before producing
more condensed and visual summaries.

In [ ]:
# ============================================================
# FIRST PLAYER PROFILE — VINICIUS JUNIOR
# ============================================================

PLAYER_NAME = "Vinicius Júnior"
PLAYER_SQUAD = "Real Madrid"
PLAYER_SEASON = "2022-2023"

mask = (
    (assignments["player"] == PLAYER_NAME)
    & (assignments["squad"] == PLAYER_SQUAD)
    & (assignments["season"] == PLAYER_SEASON)
)

matches = assignments.loc[mask]

assert len(matches) == 1, (
    "Expected exactly one matching player-club-season spell. "
    "Check the player name, squad and season."
)

player_row = matches.iloc[0]

idx = int(player_row["observation_row_idx"])

# ------------------------------------------------------------
# 1. Basic player and archetype information
# ------------------------------------------------------------

cluster_names = {
    0: "Penalty-area attacking and shooting",
    1: "Crossing-oriented wide play",
    2: "Deeper passing progression",
    3: "Direct carrying and dribbling",
    4: "Advanced creative progression"
}

cluster_id = int(player_row["k5_cluster"])

print("=" * 65)
print("WINGER PLAYER PROFILE")
print("=" * 65)

print("Player:", player_row["player"])
print("Club:", player_row["squad"])
print("Season:", player_row["season"])

if "Minutes" in assignments.columns:
    print("Minutes:", player_row["Minutes"])

print("\nProvisional archetype:")
print(f"Cluster {cluster_id} — {cluster_names[cluster_id]}")

print("\nCluster diagnostics:")
print(
    "Centroid distance:",
    round(float(player_row["k5_centroid_distance"]), 3)
)
print(
    "Silhouette:",
    round(float(player_row["k5_silhouette"]), 3)
)
print(
    "Centroid margin:",
    round(float(player_row["k5_centroid_margin"]), 3)
)

# ------------------------------------------------------------
# 2. Behavioural fingerprint
# ------------------------------------------------------------

population_percentiles = (
    raw_df[features]
    .rank(method="average", pct=True) * 100
)

fingerprint = pd.DataFrame({
    "feature": features,
    "family": family_map.loc[features].to_numpy(),
    "raw_value": X_raw[idx],
    "standardized_z": X_scaled[idx],
    "population_percentile":
        population_percentiles.iloc[idx][features].to_numpy()
})

print("\nHighest relative feature measurements:")

display(
    fingerprint.nlargest(
        8, "standardized_z"
    ).round(3)
)

print("\nLowest relative feature measurements:")

display(
    fingerprint.nsmallest(
        8, "standardized_z"
    ).round(3)
)

# ------------------------------------------------------------
# 3. Continuous PCA coordinates
# ------------------------------------------------------------

print("\nPCA coordinates:")

display(
    player_row[pc_cols].to_frame(
        name="score"
    ).round(3)
)

# ------------------------------------------------------------
# 4. Five closest distinct footballers
# ------------------------------------------------------------

player_neighbours = (
    neighbours[
        neighbours["target_row_idx"] == idx
    ]
    .sort_values("rank")
    .head(5)
)

print("\nFive closest historical style neighbours:")

display(
    player_neighbours[
        [
            "rank",
            "candidate_player",
            "candidate_squad",
            "candidate_season",
            "distance"
        ]
    ].round(3)
)

## 2.1 Initial Findings — Vinícius Júnior, 2022–23

Vinícius Júnior's Real Madrid 2022–23 spell provides
an example of an extreme direct-carrying winger profile.

The observation covers 2,823 playing minutes and
belongs to the provisional K=5 Cluster 3,
described as direct carrying and dribbling.

### Dominant Attacking Behaviours

His highest relative measurements include:

- Carries into the penalty area per 90:
  4.463, 99.86th percentile.
- Take-ons attempted per 90:
  9.756, 99.71st percentile.
- Take-ons per 100 live touches:
  17.916, 99.43rd percentile.
- Progressive receptions per 90:
  15.781, 99.50th percentile.
- Progressive carries per 90:
  7.046, 98.35th percentile.
- Attacking-third touch share:
  69.7%, 99.79th percentile.

These measurements indicate an exceptionally
carry-oriented and take-on-oriented attacking profile,
characterized by frequent advanced receiving,
ball progression and penalty-area entry.

### Relative Passing and Shooting Tendencies

Long-pass share, progressive-pass share, crossing
activity and progressive-pass volume are relatively low.

This supports a carry-led rather than
distribution-led attacking profile.

Shooting propensity is at the 20.34th percentile,
while average shot distance is at the 23.46th percentile.

The latter indicates a comparatively shorter
average shooting distance, subject to the documented
source unit.

These measurements describe action tendencies,
not shooting quality or attacking effectiveness.

### Archetype Membership

Cluster: 3 — Direct Carrying and Dribbling.

- Silhouette: 0.281.
- Centroid distance: 6.098.
- Normalized centroid margin: 0.331.

The silhouette exceeds Cluster 3's population mean,
suggesting relatively good separation from other groups.

However, the observation is far from its assigned
cluster centroid.

Thus, Vinícius represents a pronounced version
of the direct-carrying style rather than an especially
typical observation near the group average.

### Continuous Playstyle Dimensions

The first PCA score is strongly positive (PC1 = 8.314).

This is consistent with Notebook 05's interpretation
of PC1 as an axis dominated by advanced carrying
and attacking penetration.

PCA coordinates describe continuous behavioural
variation rather than discrete group membership.

### Historical Similarity

The five closest distinct historical footballers are:

1. Samuel Chukwueze, Villarreal 2021–22: 3.407.
2. Bradley Barcola, PSG 2023–24: 4.142.
3. Kingsley Coman, Bayern Munich 2017–18: 4.149.
4. Kaoru Mitoma, Brighton 2023–24: 4.430.
5. Wilfried Zaha, Crystal Palace 2017–18: 4.644.

These are Euclidean distances in the complete
standardized 20-feature space.

The nearest available distance exceeds the population
median first-neighbour distance of 1.982.

Accordingly, these footballers are relative nearest
neighbours rather than near-identical stylistic matches.

### Next Step

Develop a feature-level visualization and distinguish:

- Behaviour relative to the full winger population.
- Behaviour relative to the assigned archetype.
- Behaviour relative to alternative archetypes.
- Behaviour relative to the closest historical footballer.

The final profile must preserve both group-level
interpretability and individual stylistic variation.

In [ ]:
from matplotlib.patches import Patch

# Organize features by football behaviour family
family_order = [
    "Territory",
    "Receiving",
    "Carrying",
    "Dribbling",
    "Passing",
    "Crossing",
    "Shooting",
    "Behavioural Choice"
]

plot_data = fingerprint.copy()

plot_data["family"] = pd.Categorical(
    plot_data["family"],
    categories=family_order,
    ordered=True
)

plot_data = (
    plot_data
    .sort_values(
        ["family", "standardized_z"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

cmap = plt.get_cmap("tab10")

family_colors = {
    family: cmap(i)
    for i, family in enumerate(family_order)
}

bar_colors = [
    family_colors[str(f)]
    for f in plot_data["family"]
]

feature_labels = [
    f"{row.feature.replace('_', ' ')} "
    f"({row.population_percentile:.0f}th)"
    for row in plot_data.itertuples()
]

fig, ax = plt.subplots(figsize=(13, 11))

ax.barh(
    range(len(plot_data)),
    plot_data["standardized_z"],
    color=bar_colors,
    height=0.72
)

ax.set_yticks(range(len(plot_data)))
ax.set_yticklabels(feature_labels, fontsize=9)
ax.invert_yaxis()

ax.axvline(
    0,
    color="black",
    linewidth=1
)

ax.set_xlabel(
    "Standardized Deviation from Population Mean (Z-Score)"
)

ax.set_title(
    f"{PLAYER_NAME} — {PLAYER_SEASON}\n"
    "Complete Attacking Behavioural Fingerprint",
    fontsize=13
)

ax.grid(axis="x", alpha=0.2)

legend_handles = [
    Patch(
        color=family_colors[f],
        label=f
    )
    for f in family_order
]

ax.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.17),
    ncol=4,
    fontsize=9
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ARCHETYPE EXPLANATION:
# Assigned centroid vs nearest alternative centroid
# ============================================================

player_vector = X_scaled[idx]

centroid_distances = np.linalg.norm(
    k5_model.cluster_centers_ - player_vector,
    axis=1
)

cluster_order = np.argsort(centroid_distances)

assigned_cluster = int(cluster_order[0])
alternative_cluster = int(cluster_order[1])

assert assigned_cluster == cluster_id

assigned_centroid = (
    k5_model.cluster_centers_[assigned_cluster]
)

alternative_centroid = (
    k5_model.cluster_centers_[alternative_cluster]
)

print("=" * 65)
print("ARCHETYPE COMPARISON")
print("=" * 65)

print(
    f"Assigned: Cluster {assigned_cluster} — "
    f"{cluster_names[assigned_cluster]}"
)

print(
    f"Alternative: Cluster {alternative_cluster} — "
    f"{cluster_names[alternative_cluster]}"
)

print("\nDistances to all archetype centroids:")

centroid_ranking = pd.DataFrame({
    "cluster": range(5),
    "archetype": [
        cluster_names[i]
        for i in range(5)
    ],
    "distance": centroid_distances
}).sort_values("distance")

display(centroid_ranking.round(3))

# ------------------------------------------------------------
# Feature-level squared-distance contributions
# ------------------------------------------------------------

assigned_sq = (
    player_vector - assigned_centroid
) ** 2

alternative_sq = (
    player_vector - alternative_centroid
) ** 2

# Positive: feature favours assigned centroid
# Negative: feature favours alternative centroid
advantage = alternative_sq - assigned_sq

contrast_table = pd.DataFrame({
    "feature": features,
    "family": family_map.loc[features].to_numpy(),
    "player_z": player_vector,
    "assigned_centroid_z": assigned_centroid,
    "alternative_centroid_z": alternative_centroid,
    "assigned_squared_gap": assigned_sq,
    "alternative_squared_gap": alternative_sq,
    "advantage_for_assigned": advantage
})

np.testing.assert_allclose(
    advantage.sum(),
    centroid_distances[alternative_cluster] ** 2
    - centroid_distances[assigned_cluster] ** 2,
    atol=1e-8
)

print("\nFeatures most strongly favouring assigned archetype:")

display(
    contrast_table.nlargest(
        8, "advantage_for_assigned"
    ).round(3)
)

print("\nFeatures favouring the alternative archetype:")

display(
    contrast_table.nsmallest(
        8, "advantage_for_assigned"
    ).round(3)
)

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

chart_data = (
    contrast_table
    .assign(
        abs_advantage=lambda d:
            d["advantage_for_assigned"].abs()
    )
    .nlargest(12, "abs_advantage")
    .sort_values("advantage_for_assigned")
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    chart_data["feature"],
    chart_data["advantage_for_assigned"]
)

ax.axvline(0, color="black", linewidth=1)

ax.set_xlabel(
    "Squared-Distance Advantage: "
    "Positive = Assigned Archetype"
)

ax.set_title(
    f"{PLAYER_NAME}: Cluster {assigned_cluster} "
    f"vs Cluster {alternative_cluster}"
)

ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# NEAREST HISTORICAL FOOTBALLER:
# Feature-level comparison
# ============================================================

target_neighbours = (
    neighbours[
        neighbours["target_row_idx"] == idx
    ]
    .sort_values("rank")
)

nearest = target_neighbours.iloc[0]

candidate_idx = int(
    nearest["candidate_row_idx"]
)

candidate_vector = X_scaled[candidate_idx]

feature_differences = (
    player_vector - candidate_vector
)

squared_differences = feature_differences ** 2

computed_distance = np.sqrt(
    squared_differences.sum()
)

np.testing.assert_allclose(
    computed_distance,
    float(nearest["distance"]),
    rtol=1e-6,
    atol=1e-6
)

comparison_table = pd.DataFrame({
    "feature": features,
    "family": family_map.loc[features].to_numpy(),
    "target_z": player_vector,
    "candidate_z": candidate_vector,
    "target_minus_candidate_z": feature_differences,
    "squared_distance_contribution": squared_differences,
    "contribution_pct": (
        squared_differences
        / squared_differences.sum()
        * 100
    )
}).sort_values(
    "squared_distance_contribution",
    ascending=False
)

print("=" * 65)
print("CLOSEST HISTORICAL FOOTBALLER")
print("=" * 65)

print(
    "Target:",
    PLAYER_NAME,
    PLAYER_SEASON
)

print(
    "Neighbour:",
    nearest["candidate_player"],
    nearest["candidate_squad"],
    nearest["candidate_season"]
)

print("Euclidean distance:", round(computed_distance, 3))

print("\nLargest behavioural differences:")

display(
    comparison_table.head(10).round(3)
)

print("\nMost similar individual features:")

display(
    comparison_table.tail(8)
    .sort_values("squared_distance_contribution")
    .round(3)
)

# ------------------------------------------------------------
# Benchmark against historical nearest-neighbour distances
# ------------------------------------------------------------

first_neighbour_distances = (
    neighbours.loc[
        neighbours["rank"] == 1,
        "distance"
    ].to_numpy(dtype=float)
)

assert len(first_neighbour_distances) == len(raw_df)

# Rank the saved distances, rather than a recomputed float, to preserve ties.
nearest_distance_percentiles = (
    neighbours.loc[neighbours["rank"] == 1]
    .set_index("target_row_idx")["distance"]
    .rank(method="average", pct=True)
    .mul(100)
)
nearest_distance_percentile = float(nearest_distance_percentiles.loc[idx])

print(
    "\nNearest-neighbour distance percentile:",
    round(nearest_distance_percentile, 2)
)

print(
    "Population median first-neighbour distance:",
    round(np.median(first_neighbour_distances), 3)
)


## 3. Initial Contrastive Profile Findings

### 3.1 Behavioural Fingerprint

Vinícius Júnior's 2022–23 Real Madrid spell exhibits
exceptionally high take-on propensity, progressive
receiving and penalty-area carrying.

The profile is predominantly carry-led rather than
progressive-pass-led or crossing-led.

The distinction between shots per 90 and shots
per attacking-third touch also demonstrates why
action volume and action propensity must be
interpreted separately.

### 3.2 Comparison with the Nearest Archetype

The observation is assigned to Cluster 3
(Direct Carrying and Dribbling).

Its nearest alternative is Cluster 4
(Advanced Creative Progression).

Distances:

- Cluster 3: 6.098
- Cluster 4: 9.119

The strongest features favouring Cluster 3 include
carries into the penalty area, take-on propensity,
carry progression share and carry box-entry share.

The contrast is derived from exact differences
in squared Euclidean distance to the two centroids.

These are geometric contributions, not measures
of footballing importance or causal influence.

The player's relatively high distance from his
assigned centroid demonstrates that a clear
cluster assignment need not imply that the player
is a typical representative of the group.

### 3.3 Historical Player Similarity

The nearest distinct historical footballer is
Samuel Chukwueze, Villarreal 2021–22.

Their Euclidean distance is 3.407.

The first-neighbour distance lies at approximately
the 98.42nd percentile of the observation-level
first-neighbour distance distribution.

Vinícius therefore has an unusually distant
nearest neighbour within this historical sample.

The largest differences between the two profiles
concern:

- Progressive receptions per 90.
- Attacking-third touch share.
- Penalty-area touch share.

Together, these three features account for 61.75%
of their total squared Euclidean distance.

The profiles are relatively similar in take-on
propensity, progressive passing volume and
average shot distance.

### 3.4 Profiling Implications

A useful player report must distinguish:

1. Behavioural extremity relative to the population.
2. Similarity to an archetype centroid.
3. Separation from alternative archetypes.
4. Similarity to other actual footballers.

None of these measurements should be substituted
for the others.

Archetype labels and nearest neighbours describe
observed attacking behaviour, not player quality.

The next stage generalizes this analysis across
contrasting footballers.

In [ ]:
# ============================================================
# TWO-SIDED ARCHETYPE CONTRAST
# ============================================================

positive = (
    contrast_table[
        contrast_table["advantage_for_assigned"] > 0
    ]
    .nlargest(8, "advantage_for_assigned")
)

negative = (
    contrast_table[
        contrast_table["advantage_for_assigned"] < 0
    ]
    .sort_values("advantage_for_assigned")
)

plot_contrast = (
    pd.concat([positive, negative])
    .sort_values("advantage_for_assigned")
)

fig, ax = plt.subplots(figsize=(11, 7))

ax.barh(
    plot_contrast["feature"].str.replace("_", " ", regex=False),
    plot_contrast["advantage_for_assigned"]
)

ax.axvline(0, color="black", linewidth=1)

ax.set_xlabel(
    "Difference in Squared Centroid Distance\n"
    "Negative = Alternative | Positive = Assigned"
)

ax.set_title(
    f"{PLAYER_NAME} — Cluster {assigned_cluster} "
    f"vs Cluster {alternative_cluster}"
)

ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

print(
    "Features favouring assigned archetype:",
    (contrast_table["advantage_for_assigned"] > 0).sum()
)

print(
    "Features favouring alternative archetype:",
    (contrast_table["advantage_for_assigned"] < 0).sum()
)

In [ ]:
# ============================================================
# REUSABLE WINGER PROFILE GENERATOR
# ============================================================

# Calculate population percentiles once
all_percentiles = (
    raw_df[features]
    .rank(method="average", pct=True)
    .mul(100)
)

nearest_distance_percentiles = (
    neighbours.loc[neighbours["rank"] == 1]
    .set_index("target_row_idx")["distance"]
    .rank(method="average", pct=True)
    .mul(100)
)

def build_player_profile(player, squad, season):

    mask = (
        assignments["player"].eq(player)
        & assignments["squad"].eq(squad)
        & assignments["season"].eq(season)
    )

    matches = assignments.loc[mask]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one observation for "
            f"{player} | {squad} | {season}. "
            f"Found {len(matches)}."
        )

    row = matches.iloc[0]
    i = int(row["observation_row_idx"])

    player_vector = X_scaled[i]

    # --------------------------------------------------------
    # A. Behavioural fingerprint
    # --------------------------------------------------------

    fingerprint_df = pd.DataFrame({
        "feature": features,
        "family": family_map.loc[features].to_numpy(),
        "raw_value": X_raw[i],
        "z_score": X_scaled[i],
        "percentile": (
            all_percentiles.iloc[i][features].to_numpy()
        )
    })

    # --------------------------------------------------------
    # B. Archetype comparisons
    # --------------------------------------------------------

    distances = np.linalg.norm(
        k5_model.cluster_centers_ - player_vector,
        axis=1
    )

    ranking = np.argsort(distances)

    assigned = int(ranking[0])
    alternative = int(ranking[1])

    assert assigned == int(row["k5_cluster"])

    centroid_table = pd.DataFrame({
        "cluster": range(5),
        "archetype": [
            cluster_names[k] for k in range(5)
        ],
        "distance": distances
    }).sort_values("distance")

    assigned_sq = (
        player_vector - k5_model.cluster_centers_[assigned]
    ) ** 2

    alternative_sq = (
        player_vector - k5_model.cluster_centers_[alternative]
    ) ** 2

    archetype_contrast = pd.DataFrame({
        "feature": features,
        "family": family_map.loc[features].to_numpy(),
        "player_z": player_vector,
        "assigned_centroid_z":
            k5_model.cluster_centers_[assigned],
        "alternative_centroid_z":
            k5_model.cluster_centers_[alternative],
        "advantage_for_assigned":
            alternative_sq - assigned_sq
    })

    # --------------------------------------------------------
    # C. Historical neighbours
    # --------------------------------------------------------

    nearest_players = (
        neighbours.loc[
            neighbours["target_row_idx"] == i
        ]
        .sort_values("rank")
        .copy()
    )

    first_neighbour = nearest_players.iloc[0]
    candidate_i = int(first_neighbour["candidate_row_idx"])

    differences = player_vector - X_scaled[candidate_i]
    squared_differences = differences ** 2

    neighbour_contrast = pd.DataFrame({
        "feature": features,
        "family": family_map.loc[features].to_numpy(),
        "target_z": player_vector,
        "neighbour_z": X_scaled[candidate_i],
        "difference_z": differences,
        "squared_distance_contribution":
            squared_differences,
        "contribution_pct":
            squared_differences
            / squared_differences.sum() * 100
    }).sort_values(
        "squared_distance_contribution",
        ascending=False
    )

    first_distance = float(first_neighbour["distance"])

    np.testing.assert_allclose(
        np.sqrt(squared_differences.sum()),
        first_distance,
        rtol=1e-6,
        atol=1e-6
    )

    # --------------------------------------------------------
    # D. Player summary
    # --------------------------------------------------------

    summary = {
        "observation_row_idx": i,
        "player": player,
        "squad": squad,
        "season": season,
        "cluster": assigned,
        "archetype": cluster_names[assigned],
        "nearest_alternative_cluster": alternative,
        "silhouette": float(row["k5_silhouette"]),
        "centroid_distance": float(distances[assigned]),
        "centroid_margin": float(row["k5_centroid_margin"]),
        "nearest_player": first_neighbour["candidate_player"],
        "nearest_player_season":
            first_neighbour["candidate_season"],
        "nearest_player_distance": first_distance,
        "nearest_distance_percentile": float(
            nearest_distance_percentiles.loc[i]
        ),
        "stable_across_three_tests": bool(
            row["k5_label_preserved_all_three_tests"]
        )
    }

    # --------------------------------------------------------
    # E. Return complete analytical profile
    # --------------------------------------------------------

    return {
        "summary": summary,
        "fingerprint": fingerprint_df,
        "centroid_distances": centroid_table,
        "archetype_contrast": archetype_contrast,
        "neighbours": nearest_players,
        "neighbour_contrast": neighbour_contrast,
        "pca_scores": row[
            ["PC1", "PC2", "PC3", "PC4", "PC5"]
        ]
    }


# Validate on our already investigated example
vinicius_profile = build_player_profile(
    "Vinicius Júnior",
    "Real Madrid",
    "2022-2023"
)

assert vinicius_profile["summary"]["cluster"] == 3

np.testing.assert_allclose(
    vinicius_profile["summary"]["nearest_player_distance"],
    3.407,
    atol=0.001
)

print("Reusable player-profile function validated.")
display(pd.Series(vinicius_profile["summary"]).to_frame("value"))

In [ ]:
# ============================================================
# VALIDATE PROFILES ACROSS DIFFERENT PLAYSTYLES
# ============================================================

test_cases = []

# One representative observation for each cluster
for cluster in range(5):

    cluster_rows = assignments.loc[
        assignments["k5_cluster"] == cluster
    ]

    representative_idx = int(
        cluster_rows["k5_centroid_distance"].idxmin()
    )

    test_cases.append((
        "Cluster representative",
        representative_idx
    ))

# Previously examined extreme player
test_cases.append((
    "Extreme direct carrier",
    int(vinicius_profile["summary"]["observation_row_idx"])
))

# A borderline member of the least-separated cluster
cluster4_rows = assignments.loc[
    assignments["k5_cluster"] == 4
]

borderline_idx = int(
    cluster4_rows["k5_centroid_margin"].idxmin()
)

test_cases.append((
    "Borderline Cluster 4",
    borderline_idx
))

comparison_records = []

for case_type, row_idx in test_cases:

    observation = assignments.iloc[row_idx]

    profile = build_player_profile(
        observation["player"],
        observation["squad"],
        observation["season"]
    )

    summary = profile["summary"]

    top_features = (
        profile["fingerprint"]
        .nlargest(3, "z_score")["feature"]
        .tolist()
    )

    comparison_records.append({
        "case_type": case_type,
        "player": summary["player"],
        "squad": summary["squad"],
        "season": summary["season"],
        "cluster": summary["cluster"],
        "alternative_cluster":
            summary["nearest_alternative_cluster"],
        "silhouette": summary["silhouette"],
        "centroid_distance":
            summary["centroid_distance"],
        "centroid_margin": summary["centroid_margin"],
        "nearest_player": summary["nearest_player"],
        "nearest_distance":
            summary["nearest_player_distance"],
        "distance_percentile":
            summary["nearest_distance_percentile"],
        "stable_all_tests":
            summary["stable_across_three_tests"],
        "top_3_high_features": ", ".join(top_features)
    })

profile_comparison = pd.DataFrame(
    comparison_records
)

print("=" * 65)
print("CROSS-ARCHETYPE PLAYER PROFILE VALIDATION")
print("=" * 65)

display(
    profile_comparison.round(3)
)


## 4. Cross-Archetype Profile Validation

The reusable player-profile function was evaluated on
seven observations:

- Five observations closest to their respective
  K=5 archetype centroids.
- One extreme direct-carrying observation.
- One borderline Cluster 4 observation.

### Representative Observations

The nearest-centroid representatives are:

| Cluster | Player | Centroid Distance |
|---|---|---:|
| 0 | Victor Sá, 2019–20 | 1.421 |
| 1 | Álex Berenguer, 2021–22 | 1.483 |
| 2 | Carlos Soler, 2018–19 | 1.305 |
| 3 | Takefusa Kubo, 2022–23 | 1.278 |
| 4 | Willian, 2017–18 | 1.726 |

All five retain their original assignments across
the three previously conducted robustness experiments.

These cases demonstrate that the clustering solution
contains recognizable central representatives.

However, these observations were explicitly selected
for centroid proximity and should not be interpreted
as evidence that all cluster members are similarly
well represented.

### Extreme Profile: Vinícius Júnior

Vinícius Júnior's 2022–23 observation has:

- Cluster 3 assignment.
- Centroid distance: 6.098.
- Centroid margin: 0.331.
- Nearest historical neighbour distance: 3.407.
- Nearest-neighbour distance percentile: 98.42.

His profile is significantly more extreme than
the central Cluster 3 representative, Takefusa Kubo.

Both observations belong to the same descriptive
archetype, illustrating why cluster membership
alone is insufficient for player similarity.

### Borderline Profile: Yannis Salibur

Yannis Salibur's Guingamp 2017–18 observation has:

- Assigned cluster: 4.
- Nearest alternative: Cluster 1.
- Silhouette: -0.042.
- Centroid margin: 0.003.
- Stable across all three robustness tests: False.

This observation lies close to the boundary between
two descriptive archetypes.

Its primary assignment must therefore be presented
with the alternative centroid and ambiguity
diagnostics rather than as an exclusive football
identity.

### Implications for the Profiling Framework

A complete player report should retain:

1. Population-relative behavioural measurements.
2. The player's complete 20-feature fingerprint.
3. Their assigned archetype and centroid proximity.
4. Their closest alternative archetype.
5. Feature-level reasons for their assignment.
6. Nearest distinct historical footballers.
7. Nearest-neighbour distances and relative isolation.
8. Robustness and ambiguity diagnostics.

The archetype label is a summary of the profile,
not a replacement for the underlying measurements.

The next step is to implement a reusable visual
report that can communicate these findings for
any qualifying player-club-season observation.

In [ ]:
# ============================================================
# NOTEBOOK 08
# REUSABLE VISUAL PLAYER REPORT
# ============================================================

def display_player_report(player, squad, season):

    profile = build_player_profile(
        player,
        squad,
        season
    )

    summary = profile["summary"]

    cluster = summary["cluster"]
    alternative = summary["nearest_alternative_cluster"]

    print("=" * 75)
    print("DECODING WINGER PLAYSTYLES — PLAYER REPORT")
    print("=" * 75)

    print(f"\nPlayer: {player}")
    print(f"Club: {squad}")
    print(f"Season: {season}")

    print(
        f"\nAssigned archetype: "
        f"Cluster {cluster} — {summary['archetype']}"
    )

    print(
        f"Nearest alternative: "
        f"Cluster {alternative} — {cluster_names[alternative]}"
    )

    # --------------------------------------------------------
    # 1. Archetype diagnostics
    # --------------------------------------------------------

    cluster_population = assignments.loc[
        assignments["k5_cluster"] == cluster
    ]

    cluster_distance_percentile = float(
        cluster_population.set_index("observation_row_idx")[
            "k5_centroid_distance"
        ]
        .rank(method="average", pct=True)
        .mul(100)
        .loc[summary["observation_row_idx"]]
    )

    diagnostics = pd.DataFrame({
        "measurement": [
            "Silhouette",
            "Centroid distance",
            "Centroid margin",
            "Within-cluster centroid-distance percentile",
            "Stable across all three sensitivity tests"
        ],
        "value": [
            round(summary["silhouette"], 3),
            round(summary["centroid_distance"], 3),
            round(summary["centroid_margin"], 3),
            round(cluster_distance_percentile, 2),
            summary["stable_across_three_tests"]
        ]
    })

    print("\nARCHETYPE DIAGNOSTICS")

    display(diagnostics)

    # --------------------------------------------------------
    # 2. Fingerprint vs assigned centroid
    # --------------------------------------------------------

    fingerprint_plot = profile["fingerprint"].copy()

    fingerprint_plot["centroid_z"] = (
        k5_model.cluster_centers_[cluster]
    )

    fingerprint_plot["family"] = pd.Categorical(
        fingerprint_plot["family"],
        categories=family_order,
        ordered=True
    )

    fingerprint_plot = (
        fingerprint_plot
        .sort_values(["family", "feature"])
        .reset_index(drop=True)
    )

    y = np.arange(len(fingerprint_plot))

    fig, ax = plt.subplots(figsize=(12, 10))

    # Connect each player measurement to its centroid
    ax.hlines(
        y,
        xmin=np.minimum(
            fingerprint_plot["z_score"],
            fingerprint_plot["centroid_z"]
        ),
        xmax=np.maximum(
            fingerprint_plot["z_score"],
            fingerprint_plot["centroid_z"]
        ),
        color="0.75",
        linewidth=1.5
    )

    ax.scatter(
        fingerprint_plot["z_score"],
        y,
        s=65,
        label="Player",
        zorder=3
    )

    ax.scatter(
        fingerprint_plot["centroid_z"],
        y,
        s=65,
        marker="D",
        label=f"Cluster {cluster} centroid",
        zorder=3
    )

    # Separate feature families
    families = fingerprint_plot["family"].astype(str)

    for j in range(1, len(families)):
        if families.iloc[j] != families.iloc[j - 1]:
            ax.axhline(
                j - 0.5,
                linestyle=":",
                linewidth=0.8,
                alpha=0.5
            )

    labels = [
        f"{row.feature.replace('_', ' ')} "
        f"({row.percentile:.0f}th)"
        for row in fingerprint_plot.itertuples()
    ]

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)

    ax.invert_yaxis()

    ax.axvline(0, color="black", linewidth=1)

    ax.set_xlabel(
        "Standardized Feature Value (Z-Score)"
    )

    ax.set_title(
        f"{player} — {season}\n"
        f"Player vs Cluster {cluster} Prototype"
    )

    ax.legend()
    ax.grid(axis="x", alpha=0.2)

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # 3. Distances to all five archetypes
    # --------------------------------------------------------

    print("\nDISTANCE TO EACH ARCHETYPE")

    display(
        profile["centroid_distances"].round(3)
    )

    # --------------------------------------------------------
    # 4. Contrast with nearest alternative archetype
    # --------------------------------------------------------

    contrast = profile["archetype_contrast"]

    favours_assigned = (
        contrast.loc[
            contrast["advantage_for_assigned"] > 0
        ]
        .nlargest(5, "advantage_for_assigned")
    )

    favours_alternative = (
        contrast.loc[
            contrast["advantage_for_assigned"] < 0
        ]
        .nsmallest(5, "advantage_for_assigned")
    )

    contrast_cols = [
        "feature",
        "player_z",
        "assigned_centroid_z",
        "alternative_centroid_z",
        "advantage_for_assigned"
    ]

    print(
        f"\nFEATURES FAVOURING CLUSTER {cluster}"
    )

    display(
        favours_assigned[contrast_cols].round(3)
    )

    print(
        f"\nFEATURES FAVOURING CLUSTER {alternative}"
    )

    if len(favours_alternative):
        display(
            favours_alternative[contrast_cols].round(3)
        )
    else:
        print("No features individually favour the alternative.")

    # --------------------------------------------------------
    # 5. Nearest historical footballers
    # --------------------------------------------------------

    print("\nFIVE NEAREST HISTORICAL FOOTBALLERS")

    nearest_five = profile["neighbours"].head(5)

    display(
        nearest_five[
            [
                "rank",
                "candidate_player",
                "candidate_squad",
                "candidate_season",
                "distance"
            ]
        ].round(3)
    )

    print(
        "\nNearest-neighbour distance percentile:",
        f"{summary['nearest_distance_percentile']:.2f}"
    )

    # --------------------------------------------------------
    # 6. Return the complete profile for further analysis
    # --------------------------------------------------------

    return profile

In [ ]:
# ============================================================
# TEST CASE A — REPRESENTATIVE CROSSING-ORIENTED WINGER
# ============================================================

berenguer_profile = display_player_report(
    "Álex Berenguer",
    "Athletic Club",
    "2021-2022"
)

In [ ]:
# ============================================================
# TEST CASE B — BORDERLINE ARCHETYPE ASSIGNMENT
# ============================================================

salibur_profile = display_player_report(
    "Yannis Salibur",
    "Guingamp",
    "2017-2018"
)

In [ ]:
# Load the target summary from our most recent profile (Yannis Salibur)
summary = salibur_profile["summary"]

# Get the population for that assigned cluster
cluster_population = assignments.loc[
    assignments["k5_cluster"] == summary["cluster"]
]

# Correct within-cluster centroid-distance percentile.
# Uses the saved distances and ranks the player's own row.
cluster_distance_percentile = (
    cluster_population["k5_centroid_distance"]
    .rank(method="average", pct=True)
    .loc[summary["observation_row_idx"]
]
    * 100
)

print(f"Correct within-cluster percentile for {summary['player']}: {cluster_distance_percentile:.2f}%")


## 5. Population-Wide Player Profiling

### Objective

Convert the validated player-report methodology into
reusable datasets covering all 1,396 qualifying
player-club-season observations.

The objective is to support individual player reports,
historical comparisons and future interactive exploration
without recalculating or refitting the underlying models.

### Final Profile Components

**Player identity and historical context**

Player, club, season, minutes and observation identifier.

**Behavioural fingerprint**

All 20 attacking-behaviour features, including:

- Original feature value.
- Standardized value.
- Historical observation-level percentile.
- Assigned-cluster centroid value.

**Archetype interpretation**

- Primary descriptive cluster.
- Nearest alternative cluster.
- Distance to all five centroids.
- Silhouette.
- Centroid-distance margin.
- Within-cluster centroid-distance percentile.
- Assignment retention across robustness experiments.

**Historical similarity**

- Closest distinct footballer.
- Nearest-neighbour Euclidean distance.
- Population percentile of nearest-neighbour distance.

The full top-ten similarity table remains available
from Notebook 06.

**Continuous playstyle representation**

The first five PCA scores, retained from Notebook 05.

### Important Interpretation Rules

A high behavioural percentile is not necessarily good.

A low centroid distance indicates a more typical
archetype member, not a better footballer.

A large centroid margin indicates greater geometric
separation from the nearest alternative centroid.
It is not a membership probability.

A high nearest-neighbour distance percentile indicates
that the observation is relatively isolated within
the historical comparison population.

Feature and distance percentiles use `rank(method="average", pct=True) * 100`, including tied values. Distance displays use saved distances to preserve the same tie handling as the exported catalogue.

All percentile calculations are observation-weighted:
footballers with multiple qualifying seasons contribute
multiple observations.

The archetypes remain descriptive categories imposed
on a substantially continuous attacking-style space.

### Deliverables

1. A player-level profile catalogue.
2. A long-format behavioural-fingerprint dataset.
3. A compact CSV for inspection and application planning.
4. A metadata manifest documenting the methodology.

No statistical models will be refitted.

In [ ]:
# ============================================================
# NOTEBOOK 08 — POPULATION-WIDE PROFILE EXPORT
# ============================================================

import json
from pathlib import Path

OUTPUT_DIR = Path("/content")

n_obs = len(assignments)
n_features = len(features)

assert n_obs == 1396
assert n_features == 20

# ------------------------------------------------------------
# 1. Start with validated Notebook 07 assignments
# ------------------------------------------------------------

catalog = assignments.copy()

catalog["archetype_name"] = (
    catalog["k5_cluster"].map(cluster_names)
)

# ------------------------------------------------------------
# 2. Distances to all five cluster centroids
# ------------------------------------------------------------

centroid_distances_all = np.linalg.norm(
    X_scaled[:, None, :]
    - k5_model.cluster_centers_[None, :, :],
    axis=2
)

cluster_ids = catalog["k5_cluster"].to_numpy(dtype=int)

np.testing.assert_allclose(
    centroid_distances_all[np.arange(n_obs), cluster_ids],
    catalog["k5_centroid_distance"].to_numpy(dtype=float),
    rtol=1e-8,
    atol=1e-8
)

for cluster in range(5):
    catalog[f"distance_to_cluster_{cluster}"] = (
        centroid_distances_all[:, cluster]
    )

centroid_order = np.argsort(
    centroid_distances_all,
    axis=1
)

assert np.array_equal(
    centroid_order[:, 0],
    cluster_ids
)

alternative_clusters = centroid_order[:, 1]

catalog["alternative_cluster"] = alternative_clusters

catalog["alternative_archetype_name"] = (
    catalog["alternative_cluster"].map(cluster_names)
)

catalog["alternative_centroid_distance"] = (
    centroid_distances_all[
        np.arange(n_obs),
        alternative_clusters
    ]
)

# ------------------------------------------------------------
# 3. Correct within-cluster centroid-distance percentile
# ------------------------------------------------------------

catalog["centroid_distance_pct_within_cluster"] = (
    catalog.groupby("k5_cluster")[
        "k5_centroid_distance"
    ]
    .rank(method="average", pct=True)
    .mul(100)
)

# ------------------------------------------------------------
# 4. Attach each observation's nearest distinct footballer
# ------------------------------------------------------------

first_neighbours = (
    neighbours.loc[neighbours["rank"] == 1]
    .set_index("target_row_idx")
    .reindex(range(n_obs))
)

assert len(first_neighbours) == n_obs
assert first_neighbours["distance"].notna().all()

catalog["nearest_player"] = (
    first_neighbours["candidate_player"].to_numpy()
)

catalog["nearest_player_id"] = (
    first_neighbours["candidate_player_id"].to_numpy()
)

catalog["nearest_squad"] = (
    first_neighbours["candidate_squad"].to_numpy()
)

catalog["nearest_season"] = (
    first_neighbours["candidate_season"].to_numpy()
)

catalog["nearest_candidate_row_idx"] = (
    first_neighbours["candidate_row_idx"]
    .to_numpy(dtype=int)
)

catalog["nearest_distance"] = (
    first_neighbours["distance"].to_numpy(dtype=float)
)

catalog["nearest_distance_population_percentile"] = (
    catalog["nearest_distance"]
    .rank(method="average", pct=True)
    .mul(100)
)

# ------------------------------------------------------------
# 5. Build the 20-feature fingerprint table for each player
# ------------------------------------------------------------

family_values = (
    family_map.loc[features].to_numpy()
)

percentile_values = (
    all_percentiles[features].to_numpy(dtype=float)
)

assigned_centroid_matrix = (
    k5_model.cluster_centers_[cluster_ids]
)

fingerprint_long = pd.DataFrame({
    "observation_row_idx": np.repeat(
        catalog["observation_row_idx"].to_numpy(),
        n_features
    ),
    "feature": np.tile(features, n_obs),
    "macro_family": np.tile(family_values, n_obs),
    "raw_value": X_raw.reshape(-1),
    "z_score": X_scaled.reshape(-1),
    "population_percentile": percentile_values.reshape(-1),
    "assigned_centroid_z":
        assigned_centroid_matrix.reshape(-1)
})

# ------------------------------------------------------------
# 6. Validate both tables
# ------------------------------------------------------------

assert catalog["observation_row_idx"].is_unique

assert (
    catalog["observation_row_idx"].to_numpy()
    == np.arange(n_obs)
).all()

assert len(fingerprint_long) == n_obs * n_features

assert (
    fingerprint_long.groupby(
        "observation_row_idx"
    ).size() == n_features
).all()

np.testing.assert_allclose(
    fingerprint_long["z_score"]
    .to_numpy()
    .reshape(n_obs, n_features),
    X_scaled
)

np.testing.assert_allclose(
    fingerprint_long["raw_value"]
    .to_numpy()
    .reshape(n_obs, n_features),
    X_raw
)

assert (
    catalog["alternative_cluster"]
    != catalog["k5_cluster"]
).all()

assert (
    catalog["nearest_distance_population_percentile"]
    .between(0, 100)
).all()

assert (
    catalog["centroid_distance_pct_within_cluster"]
    .between(0, 100)
).all()

# ------------------------------------------------------------
# 7. Create a compact catalogue for browsing
# ------------------------------------------------------------

compact_columns = [
    "observation_row_idx",
    "player",
    "player_id",
    "squad",
    "season",
    "k5_cluster",
    "archetype_name",
    "alternative_cluster",
    "alternative_archetype_name",
    "k5_silhouette",
    "k5_centroid_distance",
    "k5_centroid_margin",
    "centroid_distance_pct_within_cluster",
    "k5_label_preserved_all_three_tests",
    "nearest_player",
    "nearest_squad",
    "nearest_season",
    "nearest_distance",
    "nearest_distance_population_percentile"
]

if "Minutes" in catalog.columns:
    compact_columns.insert(5, "Minutes")

compact_catalog = catalog[compact_columns].copy()

# ------------------------------------------------------------
# 8. Export the profiling datasets
# ------------------------------------------------------------

output_files = {
    "catalog":
        "winger_player_profile_catalog.parquet",
    "fingerprints":
        "winger_player_fingerprint_long.parquet",
    "compact":
        "winger_player_profile_catalog_compact.csv",
    "manifest":
        "winger_player_profiles_manifest.json"
}

catalog.to_parquet(
    OUTPUT_DIR / output_files["catalog"],
    index=False
)

fingerprint_long.to_parquet(
    OUTPUT_DIR / output_files["fingerprints"],
    index=False
)

compact_catalog.to_csv(
    OUTPUT_DIR / output_files["compact"],
    index=False
)

# ------------------------------------------------------------
# 9. Documentation manifest
# ------------------------------------------------------------

profile_manifest = {
    "project": "DECODING WINGER PLAYSTYLES",
    "notebook": "08_winger_player_profiles",

    "observational_unit": "player-club-season spell",

    "n_observations": int(n_obs),
    "n_distinct_players": int(
        catalog["player_id"].nunique()
    ),

    "feature_count": n_features,
    "feature_order": list(features),

    "feature_families": {
        feature: str(family_map.loc[feature])
        for feature in features
    },

    "primary_similarity":
        "Euclidean distance in standardized 20D space",

    "archetype_model":
        "Notebook 07 primary K-Means, K=5",

    "percentile_reference":
        "All 1396 player-club-season observations",

    "centroid_distance_percentile_reference":
        "Observations assigned to the same primary cluster",

    "nearest_distance_percentile_reference":
        "First distinct-footballer neighbour distances "
        "across all 1396 observations",

    "interpretation_limitations": [
        "Archetypes are descriptive, not natural categories.",
        "Percentiles describe behaviour, not quality.",
        "Centroid margins are not probabilities.",
        "Repeated footballers affect population percentiles.",
        "Historical nearest neighbours are not substitutes "
        "for measures of player effectiveness."
    ],

    "outputs": output_files
}

with open(
    OUTPUT_DIR / output_files["manifest"],
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        profile_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 65)
print("NOTEBOOK 08 — PROFILING EXPORTS")
print("=" * 65)

for filename in output_files.values():
    path = OUTPUT_DIR / filename
    assert path.is_file()
    print("[SAVED]", filename)

print("\nTotal player profiles:", len(catalog))
print("Total feature observations:", len(fingerprint_long))

print("\nCatalogue preview:")
display(compact_catalog.head().round(3))

In [ ]:
# ============================================================
# NOTEBOOK 08 — FINAL OUTPUT VERIFICATION
# ============================================================

test_players = [
    ("Álex Berenguer", "Athletic Club", "2021-2022"),
    ("Yannis Salibur", "Guingamp", "2017-2018"),
    ("Vinicius Júnior", "Real Madrid", "2022-2023")
]

verification_rows = []

for player, squad, season in test_players:

    match = catalog.loc[
        catalog["player"].eq(player)
        & catalog["squad"].eq(squad)
        & catalog["season"].eq(season)
    ]

    assert len(match) == 1

    row = match.iloc[0]

    verification_rows.append({
        "player": player,
        "season": season,
        "archetype": row["archetype_name"],
        "alternative": row["alternative_archetype_name"],
        "centroid_distance":
            row["k5_centroid_distance"],
        "centroid_margin":
            row["k5_centroid_margin"],
        "within_cluster_distance_pct":
            row["centroid_distance_pct_within_cluster"],
        "nearest_player": row["nearest_player"],
        "nearest_distance": row["nearest_distance"],
        "stable_all_tests":
            row["k5_label_preserved_all_three_tests"]
    })

verification = pd.DataFrame(verification_rows)

print("FINAL PLAYER-PROFILE VERIFICATION")
display(verification.round(3))

# Confirm saved files can be reloaded
reloaded_catalog = pd.read_parquet(
    OUTPUT_DIR / output_files["catalog"]
)

reloaded_fingerprints = pd.read_parquet(
    OUTPUT_DIR / output_files["fingerprints"]
)

assert len(reloaded_catalog) == 1396
assert len(reloaded_fingerprints) == 1396 * 20

assert (
    reloaded_catalog["player_id"].nunique() == 665
)

print("\n[PASS] Exported datasets reloaded successfully.")
print("[PASS] All 1,396 profiles available.")
print("[PASS] All 27,920 feature measurements available.")


## 6. Final Findings and Conclusion

### 6.1 Research Objective

Notebook 08 developed a reproducible player-profiling
framework combining the analytical representations
established in Notebooks 04–07.

The framework translates each qualifying winger's
20-dimensional attacking-behaviour representation
into a profile containing:

- Raw attacking statistics.
- Population-relative standardized measurements.
- Historical feature percentiles.
- Continuous PCA coordinates.
- Primary descriptive archetype.
- Nearest alternative archetype.
- Centroid proximity and assignment ambiguity.
- Sensitivity to alternative modelling decisions.
- Historical stylistic nearest neighbours.

The objective is to describe how footballers attack,
not to evaluate their overall quality.

### 6.2 Data and Reproducibility Validation

All required analytical artifacts were successfully
loaded and validated.

The reconstructed inputs contain:

- 1,396 player-club-season observations.
- 665 distinct footballers.
- 20 attacking-behaviour features.
- Eight behavioural feature families.

The saved K=5 model reproduces the original cluster
assignments and centroids.

All 13,960 top-ten historical neighbour distances
were successfully reproduced from the complete
standardized 20-feature representation.

No similarity result compares a footballer with
another qualifying season of the same footballer.

The profiling system reuses the established
analytical models without refitting them.

### 6.3 Individual Player Interpretation

Three contrasting player profiles were examined.

#### Álex Berenguer — Athletic Club, 2021–22

Assigned archetype:
Cluster 1 — Crossing-Oriented Wide Play.

- Centroid distance: 1.483.
- Centroid margin: 0.645.
- Within-cluster distance percentile: 0.344.
- Stable across all three robustness experiments.

High crossing volume and crossing share, together
with elevated long-pass share, support the
crossing-oriented interpretation.

This observation represents a central example
of the assigned descriptive archetype.

#### Vinícius Júnior — Real Madrid, 2022–23

Assigned archetype:
Cluster 3 — Direct Carrying and Dribbling.

- Centroid distance: 6.098.
- Centroid margin: 0.331.
- Within-cluster distance percentile: 97.368.
- Nearest distinct-footballer distance: 3.407.
- Stable across all three robustness experiments.

The player exhibits exceptionally elevated
take-on activity, progressive carrying and
penalty-area carrying.

His large centroid distance and unusually
distant nearest neighbour illustrate a
pronounced, individually distinctive profile.

He is not a typical central representative
of Cluster 3, despite a consistent assignment.

#### Yannis Salibur — Guingamp, 2017–18

Assigned archetype:
Cluster 4 — Advanced Creative Progression.

Nearest alternative:
Cluster 1 — Crossing-Oriented Wide Play.

- Assigned centroid distance: 3.498.
- Alternative centroid distance: 3.510.
- Silhouette: -0.042.
- Centroid margin: 0.003.
- Within-cluster distance percentile: 59.809.
- Stable across all three robustness experiments: No.

His most elevated individual features concern
shooting, dribbling and aspects of wide delivery.

His profile demonstrates that the nearest
archetype label may not adequately communicate
a player's most distinctive behaviours.

For such observations, alternative archetypes
and the complete feature fingerprint must
remain visible.

### 6.4 Main Research Findings

**Finding 1: Archetype membership does not imply
individual stylistic equivalence.**

Players within the same archetype can differ
substantially in behavioural extremity and
distance to their nearest footballing counterparts.

**Finding 2: Archetype typicality and assignment
ambiguity are separate properties.**

Distance to the assigned centroid indicates
how close a player is to the group's prototype.

The centroid margin indicates how much closer
the assigned prototype is than the nearest
alternative.

These measurements should not be conflated.

**Finding 3: Similarity requires full-feature
comparison.**

Historical stylistic neighbours are determined
in the complete standardized feature space.

Cluster membership provides a broader descriptive
summary but is not a substitute for individual
feature-level similarity.

**Finding 4: A single archetype label is an
incomplete player description.**

A complete profile should retain the continuous
behavioural fingerprint, PCA coordinates,
prototype comparisons and assignment diagnostics.

### 6.5 Population-Wide Profiling Outputs

The framework successfully generated four exports:

1. winger_player_profile_catalog.parquet

   Contains identity, archetype assignments,
   PCA coordinates, centroid-distance diagnostics,
   robustness indicators and nearest-neighbour data.

2. winger_player_fingerprint_long.parquet

   Contains 27,920 feature-level records,
   representing 20 measurements for each
   of the 1,396 qualifying observations.

3. winger_player_profile_catalog_compact.csv

   Provides a compact player-level dataset
   for inspection and application development.

4. winger_player_profiles_manifest.json

   Documents feature definitions, reference
   populations, analytical choices and limitations.

All four outputs were saved successfully.

The exported Parquet datasets were reloaded
and validated against their expected dimensions.

### 6.6 Methodological Limitations

The profiling framework inherits the limitations
of the underlying data and models.

- Observations represent historical player-club-season
  spells, not permanent player identities.

- The population covers qualifying winger observations
  in the five major European leagues between
  2017–18 and 2023–24.

- Feature percentiles are calculated across observations,
  not equally weighted individual footballers.

- The five K-Means groups are descriptive partitions
  within a substantially continuous style space.

- Centroid margins and silhouette values are
  geometric diagnostics, not calibrated probabilities.

- Nearest-neighbour distance measures behavioural
  resemblance, not playing ability or effectiveness.

- Differences between historical spells cannot
  automatically be attributed to tactical instruction,
  player development or individual decision-making.

- The current exported catalogue covers the observed
  historical population. Profiling an entirely new
  observation would require appropriate preprocessing
  and a compatible feature-construction pipeline.

### 6.7 Conclusion

Notebook 08 converts the research's statistical
representations into a reproducible player-level
analytical system.

The resulting framework can explain individual
attacking tendencies, identify stylistic neighbours,
describe archetype proximity and communicate
important assignment ambiguities.

The completed population-wide outputs establish
the analytical foundation for an interactive
winger exploration and comparison interface.

Notebook 08 is complete.